In [18]:
import numpy as np
import pandas as pd
import optuna
import itertools
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset e Preprocessing
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.feature_selection import SelectFromModel
from sklearn.base import BaseEstimator, RegressorMixin

# Modelli
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from catboost import CatBoostRegressor

In [19]:
#Importazione del modello
data_path = "world_happiness_report.csv"
df = pd.read_csv(data_path)

In [20]:
#info dei valori nulli
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1231 entries, 0 to 1230
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Unnamed: 0                     1231 non-null   int64  
 1   Country                        617 non-null    object 
 2   Region                         315 non-null    object 
 3   Happiness Rank                 315 non-null    float64
 4   Happiness Score                315 non-null    float64
 5   Standard Error                 158 non-null    float64
 6   Economy (GDP per Capita)       315 non-null    float64
 7   Family                         470 non-null    float64
 8   Health (Life Expectancy)       315 non-null    float64
 9   Freedom                        470 non-null    float64
 10  Trust (Government Corruption)  315 non-null    float64
 11  Generosity                     1084 non-null   float64
 12  Dystopia Residual              315 non-null    f

**Analisi info**
Da una prima analisi della info si nota che c'è una colonna senza nome che, non sapendo a cosa si riferisce, va analizzato il contenuto per eventualmente dropparla. Essendo molte le righe con valuri nulli, si analizzerà l'utilità della colonna per poi decidere come usarla, mentre per le righe con valori nulli di happiness score, considerando che sarà usata dall'algoritmo di ML, si sostituiranno quanti più valori nulli possibili

In [21]:
print(df["Unnamed: 0"])

0          0
1          1
2          2
3          3
4          4
        ... 
1226    1226
1227    1227
1228    1228
1229    1229
1230    1230
Name: Unnamed: 0, Length: 1231, dtype: int64


In [22]:
#la colonna unnamed contiene il conteggio delle colonne e potrà essere eliminata
#inoltre region ed happiness rank sono ininfluenti per l'analisi
df = df.drop(columns=["Unnamed: 0", "Region", "Happiness Rank"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1231 entries, 0 to 1230
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        617 non-null    object 
 1   Happiness Score                315 non-null    float64
 2   Standard Error                 158 non-null    float64
 3   Economy (GDP per Capita)       315 non-null    float64
 4   Family                         470 non-null    float64
 5   Health (Life Expectancy)       315 non-null    float64
 6   Freedom                        470 non-null    float64
 7   Trust (Government Corruption)  315 non-null    float64
 8   Generosity                     1084 non-null   float64
 9   Dystopia Residual              315 non-null    float64
 10  year                           1231 non-null   int64  
dtypes: float64(9), int64(1), object(1)
memory usage: 105.9+ KB


In [23]:
#valutazione media e deviazione standard
df.describe()

,Happiness Score,Standard Error,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual,year
count,315.000000,158.000000,315.000000,470.000000,315.000000,470.000000,315.000000,1084.000000,315.000000,1231.000000
mean,5.378949,0.047885,0.899837,0.990347,0.594054,0.402828,0.140532,0.153545,2.212032,2018.450041
std,1.141531,0.017146,0.410780,0.318707,0.240790,0.150356,0.115490,0.167592,0.558728,2.284034
min,2.839000,0.018480,0.000000,0.000000,0.000000,0.000000,0.000000,-0.300907,0.328580,2015.000000
25%,4.510000,0.037268,0.594900,0.793000,0.419645,0.297615,0.061315,0.064828,1.884135,2016.000000
50%,5.286000,0.043940,0.973060,1.025665,0.640450,0.418347,0.106130,0.162140,2.211260,2018.000000
75%,6.269000,0.052300,1.229000,1.228745,0.787640,0.516850,0.178610,0.252000,2.563470,2020.000000
max,7.587000,0.136930,1.824270,1.610574,1.025250,0.669730,0.551910,0.838075,3.837720,2022.000000
